# Shi Reference DIV30 Label Transfer

Standalone workflow for cell-level transfer from Shi et al. reference cells to the DIV30 AnnData object. Existing DIV30 `seurat_clusters` are used only after prediction for post hoc summaries.

In [ ]:

from __future__ import annotations

import os
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd

repo_root = Path(os.environ.get("REPO_ROOT", "/home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline")).resolve()
python_src = repo_root / "python_notebooks" / "src"
if str(python_src) not in sys.path:
    sys.path.insert(0, str(python_src))

from mge_organoid_python.paths import resolve_project_root
from mge_organoid_python.shi_label_transfer import (
    LabelTransferConfig,
    attach_reference_labels,
    comparison_label_sets,
    developmental_class_from_label,
    broad_region_from_label,
    get_umap_coordinates,
    load_shi_table_s2_labels,
    match_common_genes,
    parse_csv_set,
    plot_heatmap,
    plot_stacked_bar,
    plot_umap_categorical,
    plot_umap_continuous,
    run_knn_label_transfer,
    safe_token,
    summarize_predictions_by_cluster,
)


In [ ]:

project_root = resolve_project_root(os.environ.get("PROJECT_ROOT"))
run_label = os.environ.get("SHI_LABEL_TRANSFER_RUN_LABEL", "shi_reference_div30_label_transfer_v1")
results_dirname = os.environ.get("SHI_LABEL_TRANSFER_RESULTS_DIRNAME", "shi_reference_div30_label_transfer")
run_dir = project_root / "results" / results_dirname / run_label
plot_dir = run_dir / "plots"
table_dir = run_dir / "tables"
h5ad_dir = run_dir / "h5ad"
for directory in [run_dir, plot_dir, table_dir, h5ad_dir]:
    directory.mkdir(parents=True, exist_ok=True)

query_h5ad = Path(os.environ.get("SHI_LABEL_TRANSFER_QUERY_H5AD", project_root / "results/python_anndata/varela_div30.h5ad"))
reference_h5ad = Path(os.environ.get("SHI_LABEL_TRANSFER_REFERENCE_H5AD", project_root / "results/python_anndata/shi_2019_paper_qc.h5ad"))
shi_table_s2_xlsx = Path(os.environ.get("SHI_LABEL_TRANSFER_TABLE_S2_XLSX", project_root / "reference/shi_2021_tables_s2_to_s9/science.abj6641_table_s2.xlsx"))

config = LabelTransferConfig(
    query_h5ad=query_h5ad,
    reference_h5ad=reference_h5ad,
    shi_table_s2_xlsx=shi_table_s2_xlsx,
    run_dir=run_dir,
    n_top_variable_genes=int(os.environ.get("SHI_LABEL_TRANSFER_N_TOP_VARIABLE_GENES", "3000")),
    n_pcs=int(os.environ.get("SHI_LABEL_TRANSFER_N_PCS", "50")),
    n_neighbors=int(os.environ.get("SHI_LABEL_TRANSFER_N_NEIGHBORS", "31")),
    random_state=int(os.environ.get("SHI_LABEL_TRANSFER_RANDOM_STATE", "0")),
    n_jobs=int(os.environ.get("SLURM_CPUS_PER_TASK", os.environ.get("SHI_LABEL_TRANSFER_N_JOBS", "1"))),
    min_reference_label_match_fraction=float(os.environ.get("SHI_LABEL_TRANSFER_MIN_LABEL_MATCH_FRACTION", "0.8")),
    write_h5ad=os.environ.get("SHI_LABEL_TRANSFER_WRITE_H5AD", "1") == "1",
    save_plots=os.environ.get("SHI_LABEL_TRANSFER_SAVE_PLOTS", "1") == "1",
)

for required in [config.query_h5ad, config.reference_h5ad, config.shi_table_s2_xlsx]:
    if not required.exists():
        raise FileNotFoundError(required)

run_parameters = pd.DataFrame(
    [
        {"parameter": "project_root", "value": str(project_root)},
        {"parameter": "run_label", "value": run_label},
        {"parameter": "run_dir", "value": str(run_dir)},
        {"parameter": "query_h5ad", "value": str(config.query_h5ad)},
        {"parameter": "reference_h5ad", "value": str(config.reference_h5ad)},
        {"parameter": "shi_table_s2_xlsx", "value": str(config.shi_table_s2_xlsx)},
        {"parameter": "n_top_variable_genes", "value": config.n_top_variable_genes},
        {"parameter": "n_pcs", "value": config.n_pcs},
        {"parameter": "n_neighbors", "value": config.n_neighbors},
        {"parameter": "random_state", "value": config.random_state},
        {"parameter": "n_jobs", "value": config.n_jobs},
        {"parameter": "write_h5ad", "value": config.write_h5ad},
        {"parameter": "save_plots", "value": config.save_plots},
        {"parameter": "prediction_input_rule", "value": "DIV30 seurat_clusters are excluded from prediction and used only for post hoc summaries"},
    ]
)
run_parameters.to_csv(table_dir / "shi_label_transfer_run_parameters.tsv", sep="\t", index=False)
print("[ShiLabelTransfer] run_dir", run_dir)
print("[ShiLabelTransfer] query_h5ad", config.query_h5ad)
print("[ShiLabelTransfer] reference_h5ad", config.reference_h5ad)
print("[ShiLabelTransfer] table_s2", config.shi_table_s2_xlsx)


In [ ]:

print("[ShiLabelTransfer] loading query AnnData")
query = ad.read_h5ad(config.query_h5ad)
print("[ShiLabelTransfer] query shape", query.shape)
print("[ShiLabelTransfer] loading Shi reference AnnData")
reference = ad.read_h5ad(config.reference_h5ad)
print("[ShiLabelTransfer] reference shape", reference.shape)

if "seurat_clusters" not in query.obs.columns:
    raise ValueError("DIV30 query AnnData must contain seurat_clusters for post hoc summaries")

query_structure = pd.DataFrame(
    [
        {"object": "query", "n_obs": query.n_obs, "n_vars": query.n_vars, "obs_columns": ";".join(map(str, query.obs.columns)), "obsm_keys": ";".join(query.obsm.keys())},
        {"object": "reference", "n_obs": reference.n_obs, "n_vars": reference.n_vars, "obs_columns": ";".join(map(str, reference.obs.columns)), "obsm_keys": ";".join(reference.obsm.keys())},
    ]
)
query_structure.to_csv(table_dir / "shi_label_transfer_anndata_inputs.tsv", sep="\t", index=False)


In [ ]:

print("[ShiLabelTransfer] loading Shi Table S2 cell labels")
table_s2_labels = load_shi_table_s2_labels(config.shi_table_s2_xlsx)
table_s2_labels.to_csv(table_dir / "shi_table_s2_cell_major_type_labels.tsv", sep="\t", index=False)

reference_label_obs, label_join_summary = attach_reference_labels(
    reference,
    table_s2_labels,
    min_match_fraction=config.min_reference_label_match_fraction,
)
reference_label_obs.to_csv(table_dir / "shi_reference_table_s2_label_join.tsv", sep="\t", index=False)
label_join_summary.to_csv(table_dir / "shi_reference_table_s2_label_join_summary.tsv", sep="\t", index=False)
reference.obs["shi_table_s2_major_type"] = reference_label_obs["shi_label"].astype("string").values

label_counts = (
    reference_label_obs.dropna(subset=["shi_label"])["shi_label"]
    .value_counts()
    .rename_axis("shi_label")
    .reset_index(name="n_reference_cells")
)
label_counts["broad_region_class"] = label_counts["shi_label"].map(broad_region_from_label)
label_counts["developmental_class"] = label_counts["shi_label"].map(developmental_class_from_label)
label_counts.to_csv(table_dir / "shi_reference_table_s2_label_counts.tsv", sep="\t", index=False)
print("[ShiLabelTransfer] reference label counts")
print(label_counts.to_string(index=False))
print(label_join_summary.to_string(index=False))


In [ ]:

observed_labels = set(label_counts["shi_label"].astype(str))
non_neural_labels = parse_csv_set(os.environ.get("SHI_LABEL_TRANSFER_FULL_EXCLUDE_LABELS", "Microglia,OPC,Endothelial"))
regional_labels = parse_csv_set(os.environ.get("SHI_LABEL_TRANSFER_RESTRICTED_LABELS", "MGE,LGE,CGE"))
comparison_sets = comparison_label_sets(observed_labels, non_neural_labels=non_neural_labels, regional_labels=regional_labels)
full_override = parse_csv_set(os.environ.get("SHI_LABEL_TRANSFER_FULL_INCLUDE_LABELS"))
if full_override:
    comparison_sets["full_relevant"] = full_override

comparison_rows = []
for comparison, labels in comparison_sets.items():
    for label in sorted(labels):
        comparison_rows.append(
            {
                "comparison": comparison,
                "shi_label": label,
                "included": True,
                "broad_region_class": broad_region_from_label(label),
                "developmental_class": developmental_class_from_label(label),
            }
        )
comparison_label_table = pd.DataFrame(comparison_rows)
comparison_label_table = comparison_label_table.merge(label_counts, on=["shi_label", "broad_region_class", "developmental_class"], how="left")
comparison_label_table.to_csv(table_dir / "shi_reference_comparison_label_sets.tsv", sep="\t", index=False)
print("[ShiLabelTransfer] comparison label sets")
print(comparison_label_table.to_string(index=False))


In [ ]:

print("[ShiLabelTransfer] harmonizing genes")
query_gene_indices, reference_gene_indices, gene_detail = match_common_genes(query.var_names, reference.var_names)
gene_detail.to_csv(table_dir / "shi_label_transfer_gene_harmonization_detail.tsv", sep="\t", index=False)
gene_summary = pd.DataFrame(
    [
        {"metric": "query_genes", "value": query.n_vars},
        {"metric": "reference_genes", "value": reference.n_vars},
        {"metric": "used_common_genes", "value": int(gene_detail["used_for_transfer"].sum())},
        {"metric": "case_insensitive_matches", "value": int((gene_detail["match_type"] == "case_insensitive").sum())},
        {"metric": "ambiguous_case_insensitive_matches", "value": int((gene_detail["match_type"] == "ambiguous_case_insensitive").sum())},
    ]
)
gene_summary.to_csv(table_dir / "shi_label_transfer_gene_harmonization_summary.tsv", sep="\t", index=False)
harmonized_genes = gene_detail.loc[gene_detail["used_for_transfer"]].reset_index(drop=True)
print(gene_summary.to_string(index=False))


In [ ]:

comparison_prefix = {
    "full_relevant": "shi_full",
    "mge_lge_cge_only": "shi_mge_lge_cge",
}
all_predictions = []
reference_count_tables = []
svd_tables = []
variable_gene_tables = []
summary_tables = []
label_count_tables = []

coords, umap_key = get_umap_coordinates(query)
print("[ShiLabelTransfer] using UMAP key", umap_key)

for comparison_name, allowed_labels in comparison_sets.items():
    prefix = comparison_prefix[comparison_name]
    print(f"[ShiLabelTransfer] running {comparison_name} with labels: {sorted(allowed_labels)}")
    predictions, transfer_tables = run_knn_label_transfer(
        query=query,
        reference=reference,
        reference_label_obs=reference_label_obs,
        query_gene_indices=query_gene_indices,
        reference_gene_indices=reference_gene_indices,
        allowed_labels=allowed_labels,
        comparison_name=comparison_name,
        n_top_variable_genes=config.n_top_variable_genes,
        n_pcs=config.n_pcs,
        n_neighbors=config.n_neighbors,
        random_state=config.random_state,
        n_jobs=config.n_jobs,
    )

    renamed = predictions.rename(
        columns={
            "predicted_shi_label": f"{prefix}_predicted_shi_label",
            "prediction_score": f"{prefix}_prediction_score",
            "uncertainty_score": f"{prefix}_uncertainty_score",
            "prediction_entropy": f"{prefix}_prediction_entropy",
            "broad_region_class": f"{prefix}_broad_region_class",
            "developmental_class": f"{prefix}_developmental_class",
            "mean_neighbor_distance": f"{prefix}_mean_neighbor_distance",
            "n_neighbors_used": f"{prefix}_n_neighbors_used",
        }
    )
    for column in renamed.columns:
        if column in {"obs_name", "comparison"}:
            continue
        query.obs[column] = renamed[column].values
    all_predictions.append(renamed.reset_index(drop=True))

    ref_counts = transfer_tables["reference_label_counts"].copy()
    reference_count_tables.append(ref_counts)

    svd = transfer_tables["svd_variance"].copy()
    svd.insert(0, "comparison", comparison_name)
    svd_tables.append(svd)

    vstats = transfer_tables["variable_gene_stats"].copy()
    vstats = vstats.merge(
        harmonized_genes.reset_index().rename(columns={"index": "harmonized_gene_index"}),
        on="harmonized_gene_index",
        how="left",
    )
    vstats.insert(0, "comparison", comparison_name)
    variable_gene_tables.append(vstats)

    summaries = summarize_predictions_by_cluster(
        query.obs,
        label_col=f"{prefix}_predicted_shi_label",
        score_col=f"{prefix}_prediction_score",
        uncertainty_col=f"{prefix}_uncertainty_score",
        cluster_col="seurat_clusters",
    )
    for table_name, table in summaries.items():
        table = table.copy()
        table.insert(0, "comparison", comparison_name)
        table.to_csv(table_dir / f"div30_{prefix}_{table_name}.tsv", sep="\t", index=False)
        if table_name == "cluster_summary":
            summary_tables.append(table)
        if table_name == "label_counts":
            label_count_tables.append(table)

    if config.save_plots:
        plot_umap_categorical(
            coords,
            query.obs[f"{prefix}_predicted_shi_label"],
            f"DIV30 UMAP by Shi predicted label ({comparison_name})",
            plot_dir / f"div30_umap_{prefix}_predicted_shi_label.png",
        )
        plot_umap_continuous(
            coords,
            query.obs[f"{prefix}_prediction_score"],
            f"DIV30 UMAP by Shi prediction score ({comparison_name})",
            plot_dir / f"div30_umap_{prefix}_prediction_score.png",
        )
        plot_umap_categorical(
            coords,
            query.obs[f"{prefix}_broad_region_class"],
            f"DIV30 UMAP by broad MGE/LGE/CGE class ({comparison_name})",
            plot_dir / f"div30_umap_{prefix}_broad_region_class.png",
        )
        plot_stacked_bar(
            summaries["label_counts"],
            cluster_col="seurat_clusters",
            label_col=f"{prefix}_predicted_shi_label",
            path=plot_dir / f"div30_{prefix}_shi_label_stacked_bar_by_seurat_clusters.png",
            title=f"Shi label fractions by DIV30 seurat_clusters ({comparison_name})",
        )
        plot_heatmap(
            summaries["label_counts"],
            cluster_col="seurat_clusters",
            label_col=f"{prefix}_predicted_shi_label",
            path=plot_dir / f"div30_{prefix}_shi_label_heatmap_by_seurat_clusters.png",
            title=f"seurat_clusters x Shi predicted labels ({comparison_name})",
        )

pd.concat(all_predictions, axis=0, ignore_index=True).to_csv(table_dir / "div30_shi_label_transfer_predictions_long.tsv.gz", sep="\t", index=False)
pd.concat(reference_count_tables, axis=0, ignore_index=True).to_csv(table_dir / "shi_label_transfer_reference_cells_by_label.tsv", sep="\t", index=False)
pd.concat(svd_tables, axis=0, ignore_index=True).to_csv(table_dir / "shi_label_transfer_svd_variance.tsv", sep="\t", index=False)
pd.concat(variable_gene_tables, axis=0, ignore_index=True).to_csv(table_dir / "shi_label_transfer_variable_genes_by_comparison.tsv", sep="\t", index=False)
pd.concat(summary_tables, axis=0, ignore_index=True).to_csv(table_dir / "div30_shi_label_transfer_cluster_summaries.tsv", sep="\t", index=False)
pd.concat(label_count_tables, axis=0, ignore_index=True).to_csv(table_dir / "div30_shi_label_transfer_label_fractions_by_cluster.tsv", sep="\t", index=False)


In [ ]:

full_prefix = "shi_full"
restricted_prefix = "shi_mge_lge_cge"
comparison_obs = query.obs[
    [
        "seurat_clusters",
        f"{full_prefix}_predicted_shi_label",
        f"{full_prefix}_prediction_score",
        f"{full_prefix}_uncertainty_score",
        f"{full_prefix}_broad_region_class",
        f"{restricted_prefix}_predicted_shi_label",
        f"{restricted_prefix}_prediction_score",
        f"{restricted_prefix}_uncertainty_score",
        f"{restricted_prefix}_broad_region_class",
    ]
].copy()
comparison_obs.insert(0, "obs_name", query.obs_names.astype(str))
regional_classes = {"MGE", "LGE", "CGE"}
comparison_obs["full_prediction_is_mge_lge_cge"] = comparison_obs[f"{full_prefix}_broad_region_class"].isin(regional_classes)
comparison_obs["restricted_prediction_is_mge_lge_cge"] = comparison_obs[f"{restricted_prefix}_broad_region_class"].isin(regional_classes)
comparison_obs["same_broad_region_full_vs_restricted"] = (
    comparison_obs[f"{full_prefix}_broad_region_class"] == comparison_obs[f"{restricted_prefix}_broad_region_class"]
)
comparison_obs["same_exact_label_full_vs_restricted"] = (
    comparison_obs[f"{full_prefix}_predicted_shi_label"] == comparison_obs[f"{restricted_prefix}_predicted_shi_label"]
)
comparison_obs["restricted_forced_from_nonregional_full_prediction"] = (
    ~comparison_obs["full_prediction_is_mge_lge_cge"] & comparison_obs["restricted_prediction_is_mge_lge_cge"]
)
comparison_obs.to_csv(table_dir / "div30_shi_full_vs_mge_lge_cge_predictions.tsv.gz", sep="\t", index=False)

comparison_summary = (
    comparison_obs.assign(seurat_clusters=comparison_obs["seurat_clusters"].astype(str))
    .groupby("seurat_clusters", observed=True)
    .agg(
        n_cells=("obs_name", "size"),
        fraction_full_prediction_mge_lge_cge=("full_prediction_is_mge_lge_cge", "mean"),
        fraction_same_broad_region_full_vs_restricted=("same_broad_region_full_vs_restricted", "mean"),
        fraction_same_exact_label_full_vs_restricted=("same_exact_label_full_vs_restricted", "mean"),
        fraction_restricted_forced_from_nonregional_full_prediction=("restricted_forced_from_nonregional_full_prediction", "mean"),
        mean_full_score=(f"{full_prefix}_prediction_score", "mean"),
        mean_restricted_score=(f"{restricted_prefix}_prediction_score", "mean"),
        mean_full_uncertainty=(f"{full_prefix}_uncertainty_score", "mean"),
        mean_restricted_uncertainty=(f"{restricted_prefix}_uncertainty_score", "mean"),
    )
    .reset_index()
)
comparison_summary.to_csv(table_dir / "div30_shi_full_vs_mge_lge_cge_summary_by_seurat_clusters.tsv", sep="\t", index=False)

obs_prediction_cols = [
    "seurat_clusters",
    "orig.ident" if "orig.ident" in query.obs.columns else None,
    "cell_id" if "cell_id" in query.obs.columns else None,
    *[c for c in query.obs.columns if c.startswith("shi_full_") or c.startswith("shi_mge_lge_cge_")],
]
obs_prediction_cols = [c for c in obs_prediction_cols if c is not None and c in query.obs.columns]
obs_predictions = query.obs[obs_prediction_cols].copy()
obs_predictions.insert(0, "obs_name", query.obs_names.astype(str))
obs_predictions.to_csv(table_dir / "div30_shi_label_transfer_obs.tsv.gz", sep="\t", index=False)

if config.write_h5ad:
    output_h5ad = h5ad_dir / "div30_shi_label_transfer_predictions.h5ad"
    query.write_h5ad(output_h5ad, compression="gzip")
    print("[ShiLabelTransfer] wrote annotated h5ad", output_h5ad)
else:
    output_h5ad = None


In [ ]:

manifest_records = []
for subdir in [table_dir, plot_dir, h5ad_dir]:
    for path in sorted(subdir.rglob("*")):
        if path.is_file():
            manifest_records.append(
                {
                    "path": str(path),
                    "relative_path": str(path.relative_to(run_dir)),
                    "bytes": path.stat().st_size,
                }
            )
manifest = pd.DataFrame(manifest_records)
manifest.to_csv(table_dir / "shi_label_transfer_output_manifest.tsv", sep="\t", index=False)
completion = pd.DataFrame(
    [
        {
            "status": "complete",
            "run_dir": str(run_dir),
            "n_query_cells": query.n_obs,
            "n_reference_cells": reference.n_obs,
            "n_common_genes_used": int(gene_detail["used_for_transfer"].sum()),
            "umap_key": umap_key,
            "wrote_h5ad": bool(config.write_h5ad),
        }
    ]
)
completion.to_csv(table_dir / "shi_label_transfer_complete.tsv", sep="\t", index=False)
print("[ShiLabelTransfer] complete")
print(completion.to_string(index=False))
print("[ShiLabelTransfer] tables", table_dir)
print("[ShiLabelTransfer] plots", plot_dir)
print("[ShiLabelTransfer] h5ad", h5ad_dir)
